Load DuckDB, obviously.

In [1]:
import duckdb as db

Install and load spatial plugins

In [2]:
db.sql("INSTALL spatial; LOAD spatial;")

Load the NAPTAN stops file to get the bearings of stops

<https://beta-naptan.dft.gov.uk/download/national> download here

In [3]:
db.sql(
    """
    CREATE OR REPLACE TEMP TABLE NAPTAN AS 
    SELECT 
        ATCOCode AS stop_id, 
        Bearing
    FROM 
        read_csv_auto('../Stops.csv')""")

Load each of the components of a GTFS timetable file as a table in DuckDB

Just download a timetable from the archive and unzip it.

In [4]:
for f in ["agency", "calendar_dates", "calendar", "feed_info", "routes", "shapes", "stop_times", "stops", "trips"]:
    db.sql(f"CREATE OR REPLACE TEMP TABLE {f} AS SELECT * FROM read_csv_auto('../itm_yorkshire_gtfs_20260208/{f}.txt')")

Convert named bearing to degrees from North clockwise

In [5]:
db.sql("""
    CREATE OR REPLACE TEMP TABLE stops AS 
    SELECT *,
    CASE
        WHEN Bearing = 'N' THEN 0
        WHEN Bearing = 'NE' THEN 45
        WHEN Bearing = 'E' THEN 90
        WHEN Bearing = 'SE' THEN 135
        WHEN Bearing = 'S' THEN 180
        WHEN Bearing = 'SW' THEN 225
        WHEN Bearing = 'W' THEN 270
        WHEN Bearing = 'NW' THEN 315
        ELSE 0
    END AS numeric_bearing
    FROM stops s JOIN NAPTAN n ON s.stop_id = n.stop_id""")

Look at the stops table

In [6]:
db.sql("SELECT * FROM stops limit 1")

┌──────────────┬───────────┬───────────────────────────┬────────────────┬────────────────┬─────────────────────┬───────────────┬────────────────┬───────────────┬──────────────┬─────────┬─────────────────┐
│   stop_id    │ stop_code │         stop_name         │    stop_lat    │    stop_lon    │ wheelchair_boarding │ location_type │ parent_station │ platform_code │  stop_id_1   │ Bearing │ numeric_bearing │
│   varchar    │  varchar  │          varchar          │     double     │     double     │        int64        │     int64     │    varchar     │    varchar    │   varchar    │ varchar │      int32      │
├──────────────┼───────────┼───────────────────────────┼────────────────┼────────────────┼─────────────────────┼───────────────┼────────────────┼───────────────┼──────────────┼─────────┼─────────────────┤
│ 9400ZZSYABR1 │ 37090100  │ Arbourthorne Road To City │ 53.36337633186 │ -1.44763130301 │                   0 │             0 │ 940GZZSYABR    │ NULL          │ 9400ZZSYABR1 │ NUL

Join trips, stop_times, stops to create TIMETABLE

In [7]:
db.sql(
    """
    CREATE OR REPLACE TEMP TABLE TIMETABLE AS
    SELECT 
        * 
    FROM
        trips t
    JOIN 
        stop_times st
    ON 
        st.trip_id = t.trip_id
    JOIN 
        stops s
    ON 
        s.stop_id = st.stop_id
    """
    )

Load bus locations that are definitely in the UK

Download one of the deduped parquet files from the archive for the same day as the timetable.

In [8]:
db.sql(
    """
    CREATE OR REPLACE TEMP TABLE BUS_LOCATIONS AS 
    SELECT 
        * 
    FROM 
        '../all_bus_locations_deduplicated_20260208.parquet'
    WHERE 
        (lat BETWEEN 49 AND 59)
    AND 
        (lon BETWEEN -9 AND 2.2)
    """
)
db.sql("ALTER TABLE BUS_LOCATIONS RENAME COLUMN Bearing TO 'bus_bearing';")

Have a look at the format

In [9]:
db.sql("DESCRIBE BUS_LOCATIONS;")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ entity_id   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ trip_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ route_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ start_date  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ start_time  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ lat         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ lon         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ bus_bearing │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ timestamp   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ vehicle_id  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
├─────────────┴─────

Make an intermediate table with the distance between the bus and the stops

In [10]:
db.sql("""
    CREATE OR REPLACE TEMP TABLE ALL_BUS_TO_STOP_DISTANCES AS
    SELECT 
        T.trip_id,
        stop_id,
        arrival_time as timetabled_arrival_time,
        departure_time as timetabled_departure_time,
        timestamp as real_time,
        bus_bearing,
        numeric_bearing AS stop_bearing,
        ST_Point(lat, lon) AS bus_location,
        ST_Point(stop_lat, stop_lon) AS stop_location,
        ST_Distance_Sphere(bus_location, stop_location) AS bus_to_stop_distance_metres
    FROM TIMETABLE T JOIN BUS_LOCATIONS B ON T.trip_id = B.trip_id
    ORDER BY T.trip_id, stop_id""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In the GTFSRT data, a -1 bearing is a NULL value, so let's make it official to stop messing up our calculations

In [11]:
db.sql(
    """
    UPDATE ALL_BUS_TO_STOP_DISTANCES
    SET stop_bearing = NULL
    WHERE stop_bearing = -1
    """
)

Only consider buses that are within +-90 degrees of their stops (must be going the right way!)
This isn't perfect but it works fairly well.

In [12]:
db.sql(
    """
    CREATE OR REPLACE TEMP TABLE ALL_BUS_TO_STOP_DISTANCES_BEARINGS AS
    SELECT * FROM ALL_BUS_TO_STOP_DISTANCES
    WHERE ABS((bus_bearing - stop_bearing)) <= 90
    """
    )

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Get the bus which is closest to the stop.

In [13]:
db.sql(
    """
    CREATE OR REPLACE TEMP TABLE BUS_TO_STOP_MIN_DISTANCE AS
    SELECT
        trip_id,
        stop_id,
        ARG_MIN(timetabled_arrival_time, bus_to_stop_distance_metres) as timetabled_arrival_time,
        ARG_MIN(timetabled_departure_time, bus_to_stop_distance_metres) as timetabled_departure_time,
        ARG_MIN(strftime(to_timestamp(real_time), '%H:%M:%S'), bus_to_stop_distance_metres) as real_time_time_of_day,
        ARG_MIN(real_time, bus_to_stop_distance_metres) as real_time_timestamp,
        ARG_MIN(bus_bearing, bus_to_stop_distance_metres) as bus_bearing,
        ARG_MIN(stop_bearing, bus_to_stop_distance_metres) as stop_bearing,
        MIN(bus_to_stop_distance_metres) as distance_to_stop
    FROM ALL_BUS_TO_STOP_DISTANCES_BEARINGS
    GROUP BY trip_id, stop_id
    ORDER BY trip_id, stop_id
    """)
# this will probably be quicker without the order at the end

In [14]:
db.sql("SELECT * FROM BUS_TO_STOP_MIN_DISTANCE LIMIT 1")

┌────────────────────────────────────────────┬───────────┬─────────────────────────┬───────────────────────────┬───────────────────────┬─────────────────────┬─────────────┬──────────────┬────────────────────┐
│                  trip_id                   │  stop_id  │ timetabled_arrival_time │ timetabled_departure_time │ real_time_time_of_day │ real_time_timestamp │ bus_bearing │ stop_bearing │  distance_to_stop  │
│                  varchar                   │  varchar  │         varchar         │          varchar          │        varchar        │        int64        │   double    │    int32     │       double       │
├────────────────────────────────────────────┼───────────┼─────────────────────────┼───────────────────────────┼───────────────────────┼─────────────────────┼─────────────┼──────────────┼────────────────────┤
│ VJ0001849bcd064369d8b3be10ebefa1186e91c946 │ 450014271 │ 21:46:00                │ 21:46:00                  │ 21:43:36              │          1770587016 │      

In [15]:
print("Bus locations\n", db.sql("SELECT COUNT(*) FROM BUS_LOCATIONS"))
print("Timetable\n", db.sql("SELECT COUNT(*) FROM TIMETABLE"))
print("ALL_BUS_TO_STOP_DISTANCES\n", db.sql("SELECT COUNT(*) FROM ALL_BUS_TO_STOP_DISTANCES"))
print("BUS TO STOP MIN DISTANCE\n", db.sql("SELECT COUNT(*) FROM BUS_TO_STOP_MIN_DISTANCE"))

Bus locations
 ┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     14226639 │
└──────────────┘

Timetable
 ┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      4862036 │
└──────────────┘

ALL_BUS_TO_STOP_DISTANCES
 ┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     64769925 │
└──────────────┘

BUS TO STOP MIN DISTANCE
 ┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       559271 │
└──────────────┘



What is the furthest distance from a bus to it's stop?!

In [16]:
db.sql("SELECT bus_to_stop_distance_metres FROM ALL_BUS_TO_STOP_DISTANCES WHERE bus_to_stop_distance_metres = (SELECT MAX(bus_to_stop_distance_metres) FROM ALL_BUS_TO_STOP_DISTANCES);")

┌─────────────────────────────┐
│ bus_to_stop_distance_metres │
│           double            │
├─────────────────────────────┤
│           91889.14498484247 │
└─────────────────────────────┘